### Định lý Sturm - Tìm khoảng cách ly nghiệm


In [ ]:
import numpy as np
from IPython.display import display, Math, Markdown

def format_poly(coeffs):
    n = len(coeffs) - 1
    terms = []
    for i, c in enumerate(coeffs):
        if abs(c) < 1e-10: continue
        deg = n - i
        sign = " + " if c > 0 else " - "
        c_abs = abs(c)
        c_str = f"{c_abs:.4f}" if abs(c_abs - 1.0) > 1e-10 or deg == 0 else ""
        
        if deg == 0: terms.append(f"{sign}{c_abs:.4f}")
        elif deg == 1: terms.append(f"{sign}{c_str}x")
        else: terms.append(f"{sign}{c_str}x^{deg}")
        
    res = "".join(terms).lstrip(" + ")
    if res.startswith("- "): res = "-" + res[2:]
    return res if res else "0"

def Polynomial_Root_Isolation(coeffs_input):
    # coeffs_input là mảng hệ số đa thức, từ bậc cao nhất đến bậc 0
    # Ví dụ: x^3 - 2x + 1 => [1, 0, -2, 1]
    P = np.array(coeffs_input, dtype=float)
    n = len(P) - 1
    
    display(Markdown("## ❖ TÌM KHOẢNG CÁCH LY NGHIỆM CỦA ĐA THỨC"))
    display(Markdown("Sử dụng **Định lý Sturm** để đếm số nghiệm thực phân biệt trong khoảng [a, b]."))
    
    poly_str = format_poly(P)
    display(Math(f"P(x) = {poly_str}"))
    
    # Chuỗi Sturm
    sturm_seq = [np.poly1d(P)]
    P_deriv = np.polyder(sturm_seq[0])
    sturm_seq.append(P_deriv)
    
    md = ["### 1. Xây dựng chuỗi Sturm"]
    md.append(f"- $P_0(x) = P(x) = {format_poly(sturm_seq[0].coeffs)}$")
    md.append(f"- $P_1(x) = P'(x) = {format_poly(sturm_seq[1].coeffs)}$")
    
    while True:
        P_prev = sturm_seq[-2]
        P_curr = sturm_seq[-1]
        
        if P_curr.order == 0 and abs(P_curr.coeffs[0]) < 1e-10:
            sturm_seq.pop() # Loại bỏ đa thức 0
            break
            
        # Chia đa thức
        quot, rem = np.polydiv(P_prev, P_curr)
        # P_{i+1} = - phần dư
        P_next = np.poly1d(-rem.coeffs)
        
        if P_next.order == 0 and abs(P_next.coeffs[0]) < 1e-10:
            break
            
        sturm_seq.append(P_next)
        idx = len(sturm_seq) - 1
        md.append(f"- $P_{idx}(x) = - \\text{{dư}}(P_{idx-2}, P_{idx-1}) = {format_poly(P_next.coeffs)}$")
        
    display(Markdown('\n'.join(md)))
    
    def count_sign_changes(x_val):
        signs = []
        for p in sturm_seq:
            val = p(x_val)
            if abs(val) > 1e-10:
                signs.append(np.sign(val))
        changes = 0
        for i in range(len(signs)-1):
            if signs[i] * signs[i+1] < 0:
                changes += 1
        return changes

    # Giới hạn nghiệm theo tiêu chuẩn Cauchy
    a_n = P[0]
    A_max = np.max(np.abs(P[1:]))
    R = 1 + A_max / abs(a_n)
    
    display(Markdown("### 2. Đánh giá giới hạn nghiệm và quét tìm khoảng"))
    display(Math(f"R = 1 + \\frac{{\\max |a_k|}}{{|a_n|}} = 1 + \\frac{{{A_max:.4f}}}{{{abs(a_n):.4f}}} = {R:.4f}"))
    display(Markdown(f"Tất cả các nghiệm thực đều nằm trong khoảng $[-R, R] = [{-R:.4f}, {R:.4f}]$.\n"))
    
    intervals = []
    # Quét từ -R đến R với bước 1.0 (có thể tinh chỉnh bước quét nếu cần)
    step = 1.0
    start = -np.ceil(R)
    end = np.ceil(R)
    
    curr = start
    while curr < end:
        a = curr
        b = curr + step
        
        V_a = count_sign_changes(a)
        V_b = count_sign_changes(b)
        
        num_roots = V_a - V_b
        if num_roots == 1:
            intervals.append((a, b))
        elif num_roots > 1:
            # Nếu có >1 nghiệm trong khoảng này, chia nhỏ ra để tìm chính xác khoảng cách ly 1 nghiệm
            mid = (a + b) / 2.0
            V_mid = count_sign_changes(mid)
            if V_a - V_mid == 1: intervals.append((a, mid))
            if V_mid - V_b == 1: intervals.append((mid, b))
            
        curr += step
        
    display(Markdown("**Kết luận các khoảng cách ly nghiệm:**"))
    for i, (a, b) in enumerate(intervals):
        display(Markdown(f"- Khoảng cách ly nghiệm thứ {i+1}: $x_{i+1} \\in [{a:.4f}, {b:.4f}]$"))

# DỮ LIỆU ĐỀ BÀI (Câu 3)
# Đặc trưng đa thức của A^T A (bạn có thể lấy kết quả từ hàm Danielevski đưa vào đây)
# Ví dụ: P(lambda) = lambda^3 - 5*lambda^2 + 6*lambda
coeffs = [1.0, -5.0, 6.0, 0.0]
Polynomial_Root_Isolation(coeffs)

# ═══════════════════════════════════════════════════════════════════
# NHẬP DỮ LIỆU CỦA BẠN VÀO ĐÂY
# ═══════════════════════════════════════════════════════════════════
# He so da thuc tu bac cao nhat: a_n, ..., a_1, a_0
# Vi du: P(x) = x^3 - 6x^2 + 11x - 6
P_coeffs = [1.0, -6.0, 11.0, -6.0]
# ═══════════════════════════════════════════════════════════════════

Polynomial_Root_Isolation(P_coeffs)
